# 01 Data Preparation

This notebook prepares the mushroom images for the classification notebooks.

It keeps the data preparation simple:

1. Read the raw image folders.
2. Check that every image can be opened.
3. Remove images that are too small or blank.
4. Convert valid images to RGB JPEG.
5. Save only three CSV files: `train.csv`, `val.csv`, and `test.csv`.

The folder names inside `data/raw/mushrooms/` are used as class labels.


## 1. Settings

All paths are defined in one place. This makes the notebook easier to run locally.


In [1]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split


SEED = 42
MIN_SIZE = 32

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw" / "mushrooms"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
CLEAN_IMAGE_DIR = PROCESSED_DIR / "images"

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed directory:", PROCESSED_DIR)

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {RAW_DATA_DIR}\n"
        "Put the raw dataset in data/raw/mushrooms/."
    )

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


Project directory: c:\Users\chupr\Desktop\projects\deep-learning-minor-assignment-2
Raw data directory: c:\Users\chupr\Desktop\projects\deep-learning-minor-assignment-2\data\raw\mushrooms
Processed directory: c:\Users\chupr\Desktop\projects\deep-learning-minor-assignment-2\data\processed


## 2. Check class folders

The dataset should contain one folder per mushroom species. The class labels are taken directly from the folder names, so they are not hardcoded.


In [11]:
class_folders = sorted([path for path in RAW_DATA_DIR.iterdir() if path.is_dir()])

if len(class_folders) == 0:
    raise ValueError("No class folders found in data/raw/mushrooms/.")

label_to_id = {folder.name: index for index, folder in enumerate(class_folders)}

print("Number of classes:", len(class_folders))
for folder in class_folders:
    files = [path for path in folder.rglob("*") if path.is_file()]
    print(f"- {folder.name}: {len(files)} files")


Number of classes: 9
- Mushroom Agaricus: 353 files
- Mushroom Amanita: 750 files
- Mushroom Boletus: 1071 files
- Mushroom Cortinarius: 836 files
- Mushroom Entoloma: 364 files
- Mushroom Hygrocybe: 316 files
- Mushroom Lactarius: 1563 files
- Mushroom Russula: 1148 files
- Mushroom Suillus: 311 files


## 3. Sanitize images

This step creates a clean image copy in `data/processed/images/`.

An image is excluded if:

- it cannot be opened as an image;
- it is smaller than `32 x 32` pixels;
- it is blank.

In this notebook, a blank image means that all pixels have exactly the same RGB value. Such an image can be technically valid, but it contains no useful visual information for classification.

Valid images are converted to RGB and saved as JPEG files. This is done only to make the prepared image folder consistent and easy to read in the next notebooks.


In [12]:
def is_blank_image(image):
    image_array = np.asarray(image)

    if image_array.size == 0:
        return True

    return np.all(image_array == image_array[0, 0])


def sanitize_images():
    if CLEAN_IMAGE_DIR.exists():
        shutil.rmtree(CLEAN_IMAGE_DIR)

    CLEAN_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

    rows = []

    invalid_images = 0

    for class_folder in class_folders:
        label = class_folder.name
        label_id = label_to_id[label]

        output_folder = CLEAN_IMAGE_DIR / label
        output_folder.mkdir(parents=True, exist_ok=True)

        image_paths = sorted([path for path in class_folder.rglob("*") if path.is_file()])


        for image_path in image_paths:
            try:
                with Image.open(image_path) as image:
                    image.verify()

                with Image.open(image_path) as image:
                    image = image.convert("RGB")
                    width, height = image.size

                    if width < MIN_SIZE or height < MIN_SIZE:
                        invalid_images += 1
                        continue

                    if is_blank_image(image):
                        invalid_images += 1
                        continue

                    clean_name = f"{len(rows):06d}_{image_path.stem}.jpg"
                    clean_path = output_folder / clean_name
                    image.save(clean_path, format="JPEG", quality=95)

                    rows.append({
                        "path": clean_path.relative_to(PROJECT_DIR).as_posix(),
                        "label": label,
                        "label_id": label_id,
                        "file_name": clean_name,
                        "original_file_name": image_path.name,
                        "width": width,
                        "height": height
                    })

            except (UnidentifiedImageError, OSError, ValueError) as error:
                invalid_images += 1

    clean_df = pd.DataFrame(rows)

    return clean_df, invalid_images


clean_df, invalid_images = sanitize_images()

print("Valid images:", len(clean_df))
print("Removed images:", invalid_images)


Valid images: 6711
Removed images: 1


## 4. Class distribution

This table checks whether the prepared dataset is balanced or imbalanced after sanitizing.


In [13]:
if clean_df.empty:
    raise ValueError("No valid images were found.")

class_distribution = clean_df["label"].value_counts().sort_index()
display(class_distribution)


label
Mushroom Agaricus        353
Mushroom Amanita         750
Mushroom Boletus        1071
Mushroom Cortinarius     836
Mushroom Entoloma        364
Mushroom Hygrocybe       316
Mushroom Lactarius      1563
Mushroom Russula        1147
Mushroom Suillus         311
Name: count, dtype: int64

### Class distribution conclusion

The prepared dataset is imbalanced. The largest class is `Mushroom Lactarius`, while the smallest class is `Mushroom Suillus`.

This means that a model could become biased toward the classes with more training images.

I do not remove images from the larger classes, because this would throw away useful training data. Instead, the classification notebook will handle the imbalance with class weights. Data augmentation will also be applied only to the training set.

## 5. Train, validation and test split

The prepared images are split into:

- 70% training data;
- 15% validation data;
- 15% test data.

The split is stratified, so each subset keeps approximately the same class distribution as the full dataset.


In [14]:
train_df, temp_df = train_test_split(
    clean_df,
    test_size=0.30,
    random_state=SEED,
    stratify=clean_df["label_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_id"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

split_distribution = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts()
}).fillna(0).astype(int).sort_index()

display(split_distribution)


Train images: 4697
Validation images: 1007
Test images: 1007


,train,validation,test
label,,,
Mushroom Agaricus,247,53,53
Mushroom Amanita,525,113,112
Mushroom Boletus,749,161,161
Mushroom Cortinarius,585,126,125
Mushroom Entoloma,255,54,55
Mushroom Hygrocybe,221,47,48
Mushroom Lactarius,1094,234,235
Mushroom Russula,803,172,172
Mushroom Suillus,218,47,46


## 6. Output files

This notebook saves only the files needed by the next notebooks:

```text
data/processed/train.csv
data/processed/val.csv
data/processed/test.csv
data/processed/images/
```

No extra metadata CSV files are saved. If class weights are needed, they can be calculated later from `train.csv`.


In [15]:
print("Saved files:")
print("-", PROCESSED_DIR / "train.csv")
print("-", PROCESSED_DIR / "val.csv")
print("-", PROCESSED_DIR / "test.csv")
print("-", CLEAN_IMAGE_DIR)


Saved files:
- c:\Users\chupr\Desktop\projects\deep-learning-minor-assignment-2\data\processed\train.csv
- c:\Users\chupr\Desktop\projects\deep-learning-minor-assignment-2\data\processed\val.csv
- c:\Users\chupr\Desktop\projects\deep-learning-minor-assignment-2\data\processed\test.csv
- c:\Users\chupr\Desktop\projects\deep-learning-minor-assignment-2\data\processed\images
